# 04 — Gold: ML feature store

Fits and persists the feature pipeline (StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler),
writes the Gold feature-vector table, and stores the pipeline model in the artifacts volume
so scoring reuses the exact same encoding.

Target: `label = 1` if `arrival_delay >= 15` else `0`. Cancelled / diverted (null delay) is filtered here.

In [ ]:
import sys
sys.path.append("..")

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler,
)
from pyspark.sql.functions import col, when

from src import config

silver = spark.table(config.SILVER).filter(col("arrival_delay").isNotNull())
labeled = silver.withColumn(
    "label",
    when(col("arrival_delay") >= config.DELAY_THRESHOLD_MINUTES, 1.0).otherwise(0.0),
)

print(f"Silver rows (non-cancelled): {labeled.count():,}")
positive_rate = labeled.filter(col("label") == 1.0).count() / labeled.count()
print(f"Positive class rate: {positive_rate:.3%}")

## Feature groups

In [ ]:
categorical_cols = [
    "airline_name", "airline_code", "origin_airport_code",
    "destination_airport_code", "season",
]
boolean_cols = ["is_weekend", "is_holiday", "is_near_holiday", "is_holiday_period"]
numerical_cols = [
    "flight_month", "flight_year", "day_of_week", "week_of_year", "day_of_month",
    "quarter", "fl_number", "crs_elapsed_time", "distance", "dep_hour", "arr_hour",
    "dep_delay",
]

## Build + fit the pipeline

In [ ]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_cols
]

assembled_cols = (
    numerical_cols
    + boolean_cols
    + [f"{c}_ohe" for c in categorical_cols]
)

assembler = VectorAssembler(
    inputCols=assembled_cols,
    outputCol="features_raw",
    handleInvalid="keep",
)
scaler = StandardScaler(
    inputCol="features_raw", outputCol="features", withMean=False, withStd=True,
)

pipeline = Pipeline(stages=[*indexers, *encoders, assembler, scaler])

# Cheap fill for numerics before fitting.
prepared = labeled.na.fill(0, subset=numerical_cols + boolean_cols)
for c in categorical_cols:
    prepared = prepared.na.fill("UNKNOWN", subset=[c])

pipeline_model = pipeline.fit(prepared)
gold = pipeline_model.transform(prepared).select(
    "label", "features", *labeled.columns,
)

## Persist Gold + pipeline model

In [ ]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.GOLD)
)
gold_count = spark.table(config.GOLD).count()
print(f"Gold rows: {gold_count:,}")

pipeline_path = f"{config.ARTIFACT_VOLUME}/feature_pipeline"
pipeline_model.write().overwrite().save(pipeline_path)
print(f"Feature pipeline saved to {pipeline_path}")

## Feature manifest
The list of assembled input columns is written to a small Delta table so scoring
reconstructs the exact same feature space without hardcoding a 40-integer array.

In [ ]:
from pyspark.sql import Row

def _group_for(name: str) -> str:
    if name in numerical_cols:
        return "numeric"
    if name in boolean_cols:
        return "boolean"
    return "one_hot"

manifest_rows = [
    Row(position=i, feature_group=_group_for(c), source_column=c)
    for i, c in enumerate(assembled_cols)
]
manifest_df = spark.createDataFrame(manifest_rows)
(
    manifest_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.FEATURE_MANIFEST)
)
print(f"Wrote feature manifest ({len(manifest_rows)} rows) → {config.FEATURE_MANIFEST}")

In [ ]:
%sql
DESCRIBE HISTORY workspace.flights.gold_ml_features;